# Emotion Recognition - Model Training (Colab + GPU)

Bu notebook, GPU uzerinde tum modelleri egitip karsilastirir.

**Workflow:** GitHub Clone -> Kaggle Dataset -> GPU Training -> Results Push

**Onemli:** Runtime > Change runtime type > T4 GPU sectiginizden emin olun!

## 1. GPU Kontrolu

In [ ]:
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("[WARNING] GPU bulunamadi! Runtime > Change runtime type > T4 GPU secin.")

## 2. GitHub Repo Klonlama

In [ ]:
import os

REPO_URL = "https://github.com/aysenurhepguven0/Emotion-Aware-Adaptive-Virtual-Interaction-System.git"
REPO_NAME = "Emotion-Aware-Adaptive-Virtual-Interaction-System"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    print(f"{REPO_NAME} zaten mevcut, pull yapiliyor...")
    !cd {REPO_NAME} && git pull

%cd {REPO_NAME}
print(f"\n[OK] Calisma dizini: {os.getcwd()}")

## 3. Bagimliliklari Yukleme

In [ ]:
!pip install -q hsemotion facenet-pytorch kaggle

## 4. Kaggle API Kurulumu

**Ilk seferlik kurulum:**
1. kaggle.com/settings > API > Create New Token > kaggle.json indirilir
2. Asagidaki hucrede KAGGLE_USERNAME ve KAGGLE_KEY degerlerini doldurun

In [ ]:
import os

# ============================================
# KAGGLE BILGILERINIZI BURAYA GIRIN
# ============================================
os.environ['KAGGLE_USERNAME'] = 'KAGGLE_KULLANICI_ADINIZ'  # <-- Degistirin
os.environ['KAGGLE_KEY'] = 'KAGGLE_API_ANAHTARINIZ'        # <-- Degistirin

# Veya kaggle.json dosyasini yukleyin:
# from google.colab import files
# files.upload()  # kaggle.json dosyasini secin
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Test
!kaggle datasets list -s "fer2013" --max-size 1 2>/dev/null && print("[OK] Kaggle API calisiyor!") || print("[ERROR] Kaggle API hatasi - bilgileri kontrol edin")

## 5. Dataset Indirme (Kaggle)

Asagidaki hucrelerden ihtiyaciniz olanlari calistirin.

**Not:** Dataset adini Kaggle'daki gercek adla degistirmeniz gerekebilir.

In [ ]:
# ----- FER2013 -----
!kaggle datasets download -d msambare/fer2013 -p /tmp/fer2013 --unzip
!cp -r /tmp/fer2013/* data/fer2013/ 2>/dev/null || mkdir -p data/fer2013 && cp -r /tmp/fer2013/* data/fer2013/
print("[OK] FER2013 indirildi")
!ls data/fer2013/

In [ ]:
# ----- FER+ (FERPlus) -----
# Kaggle'daki dataset adini kontrol edin, asagidaki en yaygin olanlar:
# Secenek 1:
!kaggle datasets download -d shawon10/ferplus -p /tmp/ferplus --unzip
# Secenek 2 (eger ustteki calismazsa):
# !kaggle datasets download -d deadskull7/fer2013-augmented-ferplus -p /tmp/ferplus --unzip

!mkdir -p data/ferplus
!cp -r /tmp/ferplus/* data/ferplus/
print("[OK] FER+ indirildi")
!ls data/ferplus/

In [ ]:
# Dataset yapisini kontrol et
import os

for ds_name in ['fer2013', 'ferplus']:
    ds_path = f'data/{ds_name}'
    if os.path.exists(ds_path):
        print(f"\n{ds_name}/")
        for split in sorted(os.listdir(ds_path)):
            split_path = os.path.join(ds_path, split)
            if os.path.isdir(split_path):
                classes = [d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d))]
                total = sum(len(os.listdir(os.path.join(split_path, c))) for c in classes)
                print(f"  {split}: {total:,} images ({len(classes)} classes)")
    else:
        print(f"[ERROR] {ds_name} bulunamadi!")

## 6. Config Ayarlari (Colab icin)

GPU oldugu icin epoch sayisini artiriyoruz.

In [ ]:
# GPU varsa epoch sayisini artir
import config

if torch.cuda.is_available():
    config.EPOCHS = 30
    config.TRANSFER_BATCH_SIZE = 64  # GPU ile daha buyuk batch
    # MODEL_CONFIGS'taki batch_size'lari da guncelle
    for model_name in ['efficientnet', 'resnet', 'hsemotion']:
        config.MODEL_CONFIGS[model_name]['batch_size'] = 64
    print(f"[GPU] {config.EPOCHS} epoch, batch_size=64")
else:
    config.EPOCHS = 10
    print(f"[CPU] {config.EPOCHS} epoch")

print(f"Device: {config.DEVICE}")

## 7. Tum Modelleri Egit ve Karsilastir

Bu hucre 4 modeli sirayla egitir:
1. Mini-Xception (sifirdan)
2. EfficientNet-B0 (ImageNet pretrained)
3. ResNet-18 (ImageNet pretrained)
4. HSEmotion (AffectNet pretrained)

In [ ]:
# ============================================
# DATASET'I SECIN
# ============================================
DATASET = "ferplus"  # "fer2013", "ferplus", "rafdb", "ckplus"

!python compare_models.py --dataset {DATASET}

### 7b. (Alternatif) Tek Model Egitimi

In [ ]:
# Tek bir model egitmek isterseniz:
# !python main.py --mode train --model resnet --dataset ferplus
# !python main.py --mode train --model efficientnet --dataset ferplus
# !python main.py --mode train --model hsemotion --dataset ferplus
# !python main.py --mode train --model mini_xception --dataset ferplus

## 8. Sonuclari Incele

In [ ]:
import json
import os

# Karsilastirma sonuclarini oku
results_file = f"outputs/model_comparison_{DATASET}.json"
if os.path.exists(results_file):
    with open(results_file) as f:
        data = json.load(f)

    print("\n" + "=" * 70)
    print("  MODEL KARSILASTIRMA SONUCLARI")
    print("=" * 70)
    print(f"{'Model':<16} {'Best Val Acc':>12} {'Train Acc':>10} {'Trainable':>12} {'Sure':>10}")
    print("-" * 70)

    for r in sorted(data['results'], key=lambda x: x.get('best_val_acc', 0), reverse=True):
        if r['status'] == 'success':
            print(f"  {r['model_name']:<14} {r['best_val_acc']:>11.2f}% "
                  f"{r.get('final_train_acc', 0):>9.2f}% "
                  f"{r.get('trainable_params', 0):>11,} "
                  f"{r['train_time_formatted']:>10}")
        else:
            print(f"  {r['model_name']:<14} {'FAILED':>12}")

    print("=" * 70)
else:
    print(f"[ERROR] Sonuc dosyasi bulunamadi: {results_file}")

In [ ]:
# Training grafikleri
from IPython.display import Image, display
import glob

for img_path in sorted(glob.glob("outputs/plots/*.png")):
    print(f"\n{os.path.basename(img_path)}")
    display(Image(filename=img_path, width=600))

## 9. Sonuclari GitHub'a Push

In [ ]:
# Git config
!git config user.email "aysenurhepguven0@gmail.com"  # <-- E-posta adresinizi girin
!git config user.name "Aysenur Hep Guven"            # <-- Adinizi girin

# Sonuclari commit ve push
!git add outputs/ -f
!git status
!git commit -m "Add model comparison results ({DATASET})"

# Push (token gerekecek)
# GitHub Personal Access Token: https://github.com/settings/tokens
# !git remote set-url origin https://TOKEN@github.com/aysenurhepguven0/Emotion-Aware-Adaptive-Virtual-Interaction-System.git
# !git push origin main

## 10. Model Dosyalarini Indir (Opsiyonel)

En iyi modeli bilgisayariniza indirmek icin:

In [ ]:
from google.colab import files
import os

# En iyi modelleri listele
print("Kaydedilen modeller:")
for f in sorted(os.listdir("outputs/models/")):
    size_mb = os.path.getsize(f"outputs/models/{f}") / (1024*1024)
    print(f"  {f}: {size_mb:.1f} MB")

# Indirmek istediginiz modeli secin:
# files.download('outputs/models/best_resnet.pth')
# files.download('outputs/models/best_efficientnet.pth')
# files.download(f'outputs/model_comparison_{DATASET}.json')